In [ ]:
import os
import time
import numpy as np
import torch
import triton
import triton.language as tl

# ===============================================================
# 可选核函数：'gaussian' / 'exponential' / 'powerlaw' ---------> custom DIY kernel is also Okay
# ===============================================================
KERNEL_NAME = 'gaussian'   # 根据需要修改：'gaussian' / 'exponential' / 'powerlaw'
KERNEL_MAP  = {'gaussian': 0, 'exponential': 1, 'powerlaw': 2}
KERNEL_TYPE = KERNEL_MAP[KERNEL_NAME]

# ===============================================================
# CUDA 初始化
# ===============================================================
torch.cuda.init()
DEVICE_ID = 0
torch.cuda.set_device(DEVICE_ID)
device = torch.device(f"cuda:{DEVICE_ID}")
print(f"[INIT] 当前运行设备: {device}")

# 声学参数
res = 0.1e-3           # m，作为 a 使用
vs = 1500.0              # m/s
delta_time = 25.0e-9     # s, 采样时间间隔

# ===============================================================
# 仿真时间范围：起始下标 / 结束下标（不包含结束下标）
# ===============================================================
t_start_idx = 1400  # 包含.   如不需要，改为 0
t_end_idx   = 1900  # 不包含   如不需要，改为 采样点数
n_time_sub  = t_end_idx - t_start_idx

# ===============================================================
# 数据加载
# ===============================================================
loc_all = np.loadtxt("data/sensor_location_sphere_Fibonacci_4096.txt")  # (4096,3)
n_sensors = loc_all.shape[0]
print(f"[INFO] sensors: {n_sensors}, time samples: {n_time_sub}")

# ===============================================================
# 从 PLY 文件加载声源点数据 (21744个)
# ===============================================================
ply_path = "data/phantom_for_simulation.ply"
print(f"[Load] Reading sources from: {ply_path}")
with open(ply_path, "r") as f:
    lines = f.readlines()

start_idx = 0
for i, line in enumerate(lines):
    if line.strip() == "end_header":
        start_idx = i + 1
        break
data = np.loadtxt(lines[start_idx:])
xyz = data[:, :3]
Pc_np = data[:, 3].astype(np.float32)   # 强度 Pc

# numpy -> torch
Pc = torch.tensor(Pc_np, dtype=torch.float32, device=device).contiguous()
src_x = torch.tensor(xyz[:, 0], dtype=torch.float32, device=device).contiguous()
src_y = torch.tensor(xyz[:, 1], dtype=torch.float32, device=device).contiguous()
src_z = torch.tensor(xyz[:, 2], dtype=torch.float32, device=device).contiguous()

# 探头位置 numpy -> torch
loc_t = torch.tensor(loc_all, dtype=torch.float32, device=device)
sens_x = loc_t[:, 0].contiguous()
sens_y = loc_t[:, 1].contiguous()
sens_z = loc_t[:, 2].contiguous()



# 只为这一段时间分配输出张量 (n_sensors, n_time_sub)
out = torch.empty((n_sensors, n_time_sub), device=device, dtype=torch.float32)

# ===============================================================
# Triton 核函数：三种核可选
# ===============================================================
@triton.jit
def signal_kernel(
    Pc_ptr, src_x_ptr, src_y_ptr, src_z_ptr,
    sens_x_ptr, sens_y_ptr, sens_z_ptr,
    out_ptr,
    n_sources, n_time_sub,
    t_start_idx,
    delta_t, vs, a,
    stride_out_s, stride_out_t,
    KERNEL_TYPE: tl.constexpr,          # 0: Gaussian, 1: Exponential, 2: Powerlaw
    BLOCK_K: tl.constexpr, BLOCK_T: tl.constexpr
):
    pid_t = tl.program_id(0)  # 时间块
    pid_s = tl.program_id(1)  # 探头索引

    # 相对时间下标和掩码（只覆盖 n_time_sub）
    t_offsets = tl.arange(0, BLOCK_T)
    t_idx_rel = pid_t * BLOCK_T + t_offsets
    t_mask = t_idx_rel < n_time_sub

    # 全局时间下标 = 起始 + 相对
    t_idx_global = t_start_idx + t_idx_rel
    t_vals = t_idx_global * delta_t  # [BLOCK_T]

    # 当前探头位置
    sx = tl.load(sens_x_ptr + pid_s)
    sy = tl.load(sens_y_ptr + pid_s)
    sz = tl.load(sens_z_ptr + pid_s)

    acc = tl.zeros((BLOCK_T,), dtype=tl.float32)

    # 遍历所有声源，按 BLOCK_K 分块
    for k in range(0, n_sources, BLOCK_K):
        idx_k = k + tl.arange(0, BLOCK_K)
        src_mask = idx_k < n_sources

        px = tl.load(src_x_ptr + idx_k, mask=src_mask, other=0.0)
        py = tl.load(src_y_ptr + idx_k, mask=src_mask, other=0.0)
        pz = tl.load(src_z_ptr + idx_k, mask=src_mask, other=0.0)
        pc = tl.load(Pc_ptr    + idx_k, mask=src_mask, other=0.0)

        dx = sx - px
        dy = sy - py
        dz = sz - pz
        r = tl.sqrt(dx * dx + dy * dy + dz * dz + 1e-12)  # [BLOCK_K]

        # 扩展到 (BLOCK_K, BLOCK_T)
        r_mat  = r[:, None]          # [K,1]
        pc_mat = pc[:, None]         # [K,1]
        t_mat  = t_vals[None, :]     # [1,T]

        rt = r_mat - vs * t_mat      # [K,T]

        # 选择核函数. ----------------------------------> custom DIY kernel is also Okay
        if KERNEL_TYPE == 0:  # Gaussian
            contrib = pc_mat * 0.5 * (rt / r_mat) * tl.exp(-(rt * rt) / (2.0 * a * a))
        elif KERNEL_TYPE == 1:  # Exponential
            contrib = pc_mat * 0.5 * (rt / r_mat) * tl.exp(-tl.abs(rt) / a)
        else:  # Powerlaw: p ≈ A/(2r) * (r - v_s t) / (( (r - v_s t)^2 + a^2 )^(3/2))
            rt2 = rt * rt + a * a                    # (r - v_s t)^2 + a^2
            rt2_sqrt = tl.sqrt(rt2)                 # sqrt
            denom = r_mat * rt2 * rt2_sqrt          # r * (rt2)^(3/2) = r * rt2 * sqrt(rt2)
            contrib = (a * a * a) * pc_mat * 0.5 * rt / denom #(a * a * a)抵消中心处的放大系数

        contrib = tl.where(src_mask[:, None], contrib, 0.0)
        acc += tl.sum(contrib, axis=0)

    # 写回结果（相对时间下标存储）
    out_ptrs = out_ptr + pid_s * stride_out_s + t_idx_rel * stride_out_t
    tl.store(out_ptrs, acc, mask=t_mask)

# ===============================================================
# 启动 kernel
# ===============================================================
BLOCK_T = 128
BLOCK_K = 128
grid = (triton.cdiv(n_time_sub, BLOCK_T), n_sensors)

start = time.time()
signal_kernel[grid](
    Pc, src_x, src_y, src_z,
    sens_x, sens_y, sens_z,
    out,
    Pc.shape[0], n_time_sub,
    t_start_idx,
    delta_time, vs, res,
    out.stride(0), out.stride(1),
    KERNEL_TYPE=KERNEL_TYPE,
    BLOCK_K=BLOCK_K, BLOCK_T=BLOCK_T,
    num_warps=4, num_stages=2
)
torch.cuda.synchronize()
print(f"[TIME] Triton 前向计算耗时: {time.time() - start:.3f}s")

# ===============================================================
# 保存仿真结果为制表符分隔的文本文件
# ===============================================================
sim_output_np = out.cpu().numpy()
save_path = f"simulated_signal_{KERNEL_NAME}_t{t_start_idx}_{t_end_idx}.txt"
np.savetxt(save_path, sim_output_np, delimiter='\t')
print(f"[SAVE] 仿真结果已保存到: {save_path}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

KERNEL_NAME = 'gaussian'   # 根据需要修改：'gaussian' / 'exponential' / 'powerlaw'
t_start_idx = 1400  # 包含
t_end_idx   = 1900  # 不包含
delta_time = 25.0e-9     # s, 采样时间间隔
# 读取文件
save_path = f"simulated_signal_{KERNEL_NAME}_t{t_start_idx}_{t_end_idx}.txt"
sim_data = np.loadtxt(save_path, delimiter='\t')  # 形状: (n_sensors, n_time_sub)

sensor_id = 3142          # 想看的探头编号
sim_signal = sim_data[sensor_id]

# 构造时间轴（与仿真时的 t_start_idx / t_end_idx 对应）
t_axis_us = np.arange(t_start_idx, t_end_idx) * delta_time * 1e6  # 微秒

plt.figure(figsize=(10, 4))
plt.plot(t_axis_us, sim_signal, label=f"Sensor #{sensor_id}")
plt.xlabel("Time (µs)")
plt.ylabel("Pressure")
plt.title(f"Sensor #{sensor_id} simulated signal")
plt.legend()
plt.tight_layout()
plt.show()